# Task 1: Data Profiling and Missing Values (~25 minutes)


In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

#### 1.1 — Profile the raw data


In [19]:
df_=df.copy()
print(f"Rows and columns:\n{df_.shape}")
print(f"\nMemory usage:\n{df_.memory_usage(deep=True).sum()}")
print(f"\nMissing values:\n{df_.isnull().sum()}")
print(f"\nMissing percentage:\n:{(df_.isnull().sum() / len(df_)) * 100}")
print(f"\nUnique values:\n:{df_.nunique()}")
print(f"\nNumeric statistics:{df_.describe()}")
print(f"\nMedian:\n{df_.median(numeric_only=True)}")

Rows and columns:
(541909, 8)

Memory usage:
110092302

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Missing percentage:
:InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

Unique values:
:InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

Numeric statistics:            Quantity                 InvoiceDate      UnitPrice     CustomerID
count  541909.000000                      541909  541909.000000  406829.000000
mean        9.552250  2011-07-04 13:34:57.156386       4.611114   15287.690570
min    -80995.000000         2010-12-01 08:26:0

#### 1.2 — Classify the missing data types

In [3]:
with_id = df_[df_["CustomerID"].notnull()]
without_id = df_[df_["CustomerID"].isnull()]
print("for not nan values:\n",with_id["Country"].value_counts(),'\n')
print("\nfor nan values:\n",without_id["Country"].value_counts(),'\n')
print("\nQuantity for not nan values:\n",with_id["Quantity"].describe(),'\n')
print("\nQuantity for nan values:\n",without_id["Quantity"].describe())
print("\nUnitPrice for not nan values:\n",with_id["UnitPrice"].describe())
print("\nUnitPrice for nan values:\n",without_id["UnitPrice"].describe())


missing_desc = df_[df_["Description"].isnull()]
print('\n',missing_desc[["StockCode", "Description"]].head())
code = missing_desc.iloc[0]["StockCode"]
print('\n',df_[df_["StockCode"] == code][["StockCode", "Description"]])

for not nan values:
 Country
United Kingdom          361878
Germany                   9495
France                    8491
EIRE                      7485
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               1877
Portugal                  1480
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
USA                        291
Israel                     250
Unspecified                244
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58
Lebanon   

#### 1.3 — Handle the missing values

In [4]:
# CustomerID is unique, so it cannot be guessed, that is why rows with missing CustomerID are removed
df_ = df_.dropna(subset=["CustomerID"])

# Find description for each product using StockCode
desc_map = df_.dropna(subset=["Description"]).groupby("StockCode")["Description"].first()

# Fill missing descriptions using the matching StockCode
df_["Description"] = df_["Description"].fillna(df_["StockCode"].map(desc_map))

# If there are still missing descriptions, remove those rows too
df_ = df_.dropna(subset=["Description"])

# Check remaining missing values
print(df_[["CustomerID", "Description"]].isnull().sum())

CustomerID     0
Description    0
dtype: int64


#### Notes

For **CustomerID**, the missing values look like **MAR (Missing At Random)** because rows with missing CustomerID are not similar to rows where CustomerID exists. When I compared them, I saw differences in **Country**, **Quantity**, and **UnitPrice**. For example, rows without CustomerID are mostly from the United Kingdom, the average quantity is much lower, and the average unit price is higher. This shows that the missingness is related to other observed columns, so it is not completely random.

For handling this column, I chose **deletion**. CustomerID is a unique identifier, so it cannot be guessed or filled correctly. Creating artificial customer IDs could create wrong customer relationships in later analysis, so removing rows with missing CustomerID is a safer choice.

For **Description**, the missing values also look like **MAR** because most rows still have a valid **StockCode**, and the same StockCode in other rows usually contains a valid product description. This means the missing description can be recovered using existing information from the dataset.

For handling Description, I chose **imputation**. I used StockCode to find the first available description for the same product and filled the missing values with that description. This keeps useful rows in the dataset instead of removing them.

# Task 2: Cleaning Invalid and Anomalous Records


#### 2.1 — Identify cancellations

In [5]:
# Invoices starting with C mean cancelled transactions
cancellations = df_[df_["InvoiceNo"].astype(str).str.startswith("C")]
# Count cancellation rows
print("Number of cancellation transactions:", len(cancellations))

Number of cancellation transactions: 8905


I would remove cancellation transactions for the main analysis because they do not represent completed purchases. In this dataset, invoices starting with **"C"** have negative quantities, which means the products were returned or the order was cancelled.

If these rows stay in the dataset, they can distort sales patterns, customer spending, and product popularity. For example, in a recommender system, cancelled products should not be treated as customer preference.

For customer churn prediction, cancellations can still be useful because frequent cancellations may show customer dissatisfaction. In that case, they can be flagged as a separate feature instead of removed.

#### 2.2 — Handle invalid quantities and prices

In [6]:
invalid_qty = df_[(df_["Quantity"] <= 0) & (~df_["InvoiceNo"].astype(str).str.startswith("C"))]
invalid_price = df_[df_["UnitPrice"] <= 0]

print("Invalid quantity count:", len(invalid_qty))
print("Invalid price count:", len(invalid_price))

Invalid quantity count: 0
Invalid price count: 40


There are no non-cancellation records with invalid quantity, so no action is needed for Quantity in this category.

For UnitPrice, there are 40 records where the value is less than or equal to zero. These rows were removed because price should normally be positive in retail transactions. Zero or negative prices may represent data entry errors, free items, or non-standard transactions, and they can affect later analysis such as spending patterns or product value estimation.

In [7]:
# Find outliers in Quantity using IQR method
q1 = df_["Quantity"].quantile(0.25)
q3 = df_["Quantity"].quantile(0.75)
iqr = q3 - q1

# Define upper limit for outliers
upper = q3 + 1.5 * iqr

# Keep rows above upper limit
outliers_qty = df_[df_["Quantity"] > upper]
print("Quantity outliers:", len(outliers_qty))


# Find outliers in UnitPrice using same method
q1 = df_["UnitPrice"].quantile(0.25)
q3 = df_["UnitPrice"].quantile(0.75)
iqr = q3 - q1

# Define upper limit for price outliers
upper = q3 + 1.5 * iqr

# Keep rows above upper limit
outliers_price = df_[df_["UnitPrice"] > upper]
print("UnitPrice outliers:", len(outliers_price))

Quantity outliers: 25656
UnitPrice outliers: 36051


Extreme outliers in Quantity and UnitPrice were detected using the IQR method. I kept these records because very large orders or expensive products can still represent real customer behaviour in retail data. Removing them directly could also remove valid business information.

#### 2.3 — Clean and validate

In [8]:
# Save shape before cleaning
print("Shape before cleaning:", df_.shape)

# Remove cancellation invoices
df_ = df_[~df_["InvoiceNo"].astype(str).str.startswith("C")]

# Remove rows where price is zero or negative
df_ = df_[df_["UnitPrice"] > 0]

# Remove rows where quantity is zero or negative
df_ = df_[df_["Quantity"] > 0]

# Check final shape
print("Shape after cleaning:", df_.shape)

# Validation checks
print("Negative quantity left:", (df_["Quantity"] <= 0).sum())
print("Zero or negative price left:", (df_["UnitPrice"] <= 0).sum())

Shape before cleaning: (406829, 8)
Shape after cleaning: (397884, 8)
Negative quantity left: 0
Zero or negative price left: 0


# Task 3: Categorical Data Challenges


#### 3.1 — Analyze the Country column


In [9]:
# How many unique countries are there
print("Number of unique countries:", df_["Country"].nunique())

# Count transactions by country
country_counts = df_["Country"].value_counts()
print("\nTransactions by country:\n", country_counts)

# Percentage of transactions from top 5 countries
top5_percentage = (country_counts.head(5).sum() / len(df_)) * 100
print("\nPercentage of transactions from top 5 countries:", top5_percentage)

# How many countries have fewer than 50 transactions
rare_countries = country_counts[country_counts < 50]
print("\nNumber of countries with fewer than 50 transactions:", len(rare_countries))

# Create cleaned Country column
df_["Country_cleaned"] = df_["Country"].apply(
    lambda x: "Other" if country_counts[x] < 50 else x
)

# Compare number of categories before and after
print("\nNumber of categories before cleaning:", df_["Country"].nunique())
print("Number of categories after cleaning:", df_["Country_cleaned"].nunique())

Number of unique countries: 37

Transactions by country:
 Country
United Kingdom          354321
Germany                   9040
France                    8341
EIRE                      7236
Spain                     2484
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1462
Australia                 1182
Norway                    1071
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     248
Unspecified                244
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA 

#### 3.2 — Analyze the StockCode column


In [10]:
# Count unique stock codes
print("Number of unique StockCode:", df_["StockCode"].nunique())

# Find possible non-product codes (codes without any digit)
non_product_codes = df_[~df_["StockCode"].astype(str).str.contains(r"\d", na=False)]

# Show unusual codes
print("Possible non-product codes:", non_product_codes["StockCode"].unique())

# Remove non-product codes and keep only product-related codes
df_ = df_[df_["StockCode"].astype(str).str.contains(r"\d", na=False)]

Number of unique StockCode: 3665
Possible non-product codes: ['POST' 'M' 'BANK CHARGES' 'PADS' 'DOT']


#### 3.3 — Engineer a feature from Description


In [11]:
# Create a flag for descriptions containing the word SET
df_["has_set"] = df_["Description"].astype(str).str.contains("SET", case=False, na=False)

# Compare average Quantity and UnitPrice for products with and without SET
relationship = df_.groupby("has_set")[["Quantity", "UnitPrice"]].mean()

# Show result
print(relationship)

          Quantity  UnitPrice
has_set                      
False    13.285184   2.862660
True     10.979972   3.041122


I created a feature called **has_set** by checking whether the word **"SET"** appears in the product description.

The comparison shows that products containing **"SET"** have a slightly higher average UnitPrice (3.04 vs 2.84), while their average Quantity is lower (10.98 vs 13.29). This suggests that grouped products are usually sold at a slightly higher price but in smaller quantities.

# Task 4: Class Imbalance — Predicting High-Value Customers


#### 4.1 — Engineer a binary target


In [12]:
# Create customer-level aggregated dataset
customer_data = df_.groupby("CustomerID").agg({
    "InvoiceNo": "nunique",          # number of orders
    "StockCode": "nunique",          # distinct products
    "InvoiceDate": ["min", "max"]    # first and last purchase
})

# Flatten column names
customer_data.columns = ["num_orders", "distinct_products", "first_purchase", "last_purchase"]

# Add total revenue
customer_data["total_revenue"] = df_.groupby("CustomerID").apply(
    lambda x: (x["Quantity"] * x["UnitPrice"]).sum()
)

# Reset index
customer_data = customer_data.reset_index()

# Find top 10% threshold
threshold = customer_data["total_revenue"].quantile(0.9)

# Create binary target
customer_data["high_value"] = (customer_data["total_revenue"] >= threshold).astype(int)

# Show result
customer_data.head()

,CustomerID,num_orders,distinct_products,first_purchase,last_purchase,total_revenue,high_value
0,12346.0,1,1,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,1
1,12347.0,7,103,2010-12-07 14:57:00,2011-12-07 15:52:00,4310.00,1
2,12348.0,4,21,2010-12-16 19:09:00,2011-09-25 13:13:00,1437.24,0
3,12349.0,1,72,2011-11-21 09:51:00,2011-11-21 09:51:00,1457.55,0
4,12350.0,1,16,2011-02-02 16:01:00,2011-02-02 16:01:00,294.40,0


#### 4.2 — Measure the imbalance


In [13]:
# Class distribution
print(customer_data["high_value"].value_counts())

# Percentage distribution
print(customer_data["high_value"].value_counts(normalize=True) * 100)

# Baseline accuracy if always predicting regular
baseline_accuracy = (customer_data["high_value"] == 0).mean()
print("Baseline accuracy:", baseline_accuracy)

high_value
0    3900
1     434
Name: count, dtype: int64
high_value
0    89.986156
1    10.013844
Name: proportion, dtype: float64
Baseline accuracy: 0.8998615597600369


The dataset is imbalanced because most customers are regular customers. There are **3900 regular customers (89.99%)** and **434 high-value customers (10.01%)**.

If a model always predicts **regular**, it would still get about **89.99% accuracy**, because regular customers are the majority.

This shows that accuracy is not a very good metric here, because the model may seem accurate even if it does not identify high-value customers properly.

#### 4.3 — Apply resampling


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Select features and target
X = customer_data[["total_revenue", "num_orders", "distinct_products"]]
y = customer_data["high_value"]

# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Show class distribution before resampling
print("Before resampling:")
print(y_train.value_counts())

# Combine train data for resampling
train_data = X_train.copy()
train_data["high_value"] = y_train

majority = train_data[train_data["high_value"] == 0]
minority = train_data[train_data["high_value"] == 1]

# Oversampling minority class
minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled = pd.concat([majority, minority_oversampled])

print("\nAfter oversampling:")
print(oversampled["high_value"].value_counts())

# Undersampling majority class
majority_undersampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_undersampled, minority])

print("\nAfter undersampling:")
print(undersampled["high_value"].value_counts())

Before resampling:
high_value
0    3120
1     347
Name: count, dtype: int64

After oversampling:
high_value
0    3120
1    3120
Name: count, dtype: int64

After undersampling:
high_value
0    347
1    347
Name: count, dtype: int64


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Oversampling model
X_over = oversampled.drop("high_value", axis=1)
y_over = oversampled["high_value"]

model_over = LogisticRegression(max_iter=1000)
model_over.fit(X_over, y_over)

pred_over = model_over.predict(X_test)

print("Oversampling results:")
print(classification_report(y_test, pred_over))

# Undersampling model
X_under = undersampled.drop("high_value", axis=1)
y_under = undersampled["high_value"]

model_under = LogisticRegression(max_iter=1000)
model_under.fit(X_under, y_under)

pred_under = model_under.predict(X_test)

print("\nUndersampling results:")
print(classification_report(y_test, pred_under))

Oversampling results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       780
           1       0.99      1.00      0.99        87

    accuracy                           1.00       867
   macro avg       0.99      1.00      1.00       867
weighted avg       1.00      1.00      1.00       867


Undersampling results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       780
           1       0.98      1.00      0.99        87

    accuracy                           1.00       867
   macro avg       0.99      1.00      0.99       867
weighted avg       1.00      1.00      1.00       867



# Task 5: Data Leakage — Introduce, Detect, and Fix


#### 5.1 — Intentionally introduce temporal leakage


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Make sure InvoiceDate is datetime
df_["InvoiceDate"] = pd.to_datetime(df_["InvoiceDate"])

# Create customer-level features from the full date range (this is intentionally wrong)
customer_full = df_.groupby("CustomerID").agg({
    "InvoiceNo": "nunique",
    "StockCode": "nunique",
    "InvoiceDate": ["min", "max"]
}).reset_index()

# Fix column names
customer_full.columns = ["CustomerID", "num_orders", "distinct_products", "first_purchase", "last_purchase"]

# Add total revenue
customer_revenue = df_.groupby("CustomerID").apply(
    lambda x: (x["Quantity"] * x["UnitPrice"]).sum()
).reset_index(name="total_revenue")

customer_full = customer_full.merge(customer_revenue, on="CustomerID")

# Create target: did customer purchase in December 2011
dec_2011_customers = df_[
    (df_["InvoiceDate"].dt.year == 2011) &
    (df_["InvoiceDate"].dt.month == 12)
]["CustomerID"].unique()

customer_full["bought_dec_2011"] = customer_full["CustomerID"].isin(dec_2011_customers).astype(int)

# Select features and target
X = customer_full[["total_revenue", "num_orders", "distinct_products"]]
y = customer_full["bought_dec_2011"]

# Random split (intentionally wrong because it ignores time)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print("Leaky model performance:")
print(classification_report(y_test, y_pred))

Leaky model performance:
              precision    recall  f1-score   support

           0       0.88      0.99      0.93       744
           1       0.73      0.20      0.31       123

    accuracy                           0.88       867
   macro avg       0.80      0.59      0.62       867
weighted avg       0.86      0.88      0.84       867



The leaky model gives relatively high overall accuracy (**88%**), but the performance for the positive class is much weaker.

For customers who purchased in December 2011, precision is **0.73**, but recall is only **0.20**, which means the model misses many actual positive cases.

This happens because the features were created using the full date range, including future information. As a result, the model benefits from temporal leakage and the performance looks stronger than it would in a real forecasting setup.

#### 5.2 — Detect the leakage


In [17]:
customer_full[["total_revenue", "num_orders", "distinct_products", "bought_dec_2011"]].corr()

,total_revenue,num_orders,distinct_products,bought_dec_2011
total_revenue,1.000000,0.550686,0.381084,0.192557
num_orders,0.550686,1.000000,0.691517,0.347698
distinct_products,0.381084,0.691517,1.000000,0.284062
bought_dec_2011,0.192557,0.347698,0.284062,1.000000


Here the main issue is that train and test data are not really separated by time. Since all customer features were calculated from the full dataset first, both sets already contain information from the same months, including the period we want to predict.

Because of that, the model gets access to future behaviour, so the accuracy looks better than it normally should.

For example, total revenue becomes suspiciously informative, because it already includes purchases from the target period. In a real prediction case, we would not know that information yet, so this creates temporal leakage.

The correlations also support this idea: **num_orders** has the strongest correlation with the target (**0.35**), followed by **distinct_products (0.28)** and **total_revenue (0.19)**, which suggests that these features already contain useful future-related information.

#### 5.3 — Fix with a correct temporal split


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Temporal split
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df_[df_["InvoiceDate"] <= observation_end]
df_pred = df_[df_["InvoiceDate"] >= prediction_start]

# Create customer features only from observation window
customer_obs = df_obs.groupby("CustomerID").agg({
    "InvoiceNo": "nunique",
    "StockCode": "nunique"
}).reset_index()

customer_obs.columns = ["CustomerID", "num_orders", "distinct_products"]

# Add total revenue
customer_revenue = df_obs.groupby("CustomerID").apply(
    lambda x: (x["Quantity"] * x["UnitPrice"]).sum()
).reset_index(name="total_revenue")

customer_obs = customer_obs.merge(customer_revenue, on="CustomerID")

# Create target from prediction window
future_customers = df_pred["CustomerID"].unique()

customer_obs["future_purchase"] = customer_obs["CustomerID"].isin(future_customers).astype(int)

# Features and target
X = customer_obs[["total_revenue", "num_orders", "distinct_products"]]
y = customer_obs["future_purchase"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Temporal split model performance:")
print(classification_report(y_test, y_pred))

Temporal split model performance:
              precision    recall  f1-score   support

           0       0.66      0.77      0.71       354
           1       0.73      0.61      0.67       367

    accuracy                           0.69       721
   macro avg       0.70      0.69      0.69       721
weighted avg       0.70      0.69      0.69       721



With the correct temporal split, the model performance becomes more balanced and realistic. Accuracy drops to **69%**, which is lower than the leaky model, but this result is more trustworthy because only past behaviour is used for prediction.

Compared to the leaky model, recall for the positive class improves a lot (**0.61 vs 0.20**), which means the model now identifies future purchasers much better.

This shows that the temporal split removes leakage and gives a more realistic estimate of how the model would perform in a real prediction setting.